# FlyQuant-1 — C++ connectome-reservoir training on Google Colab

Colab is used only as a compute host. All research logic, feature engineering, training, evaluation, serialization, and inference live in the C++20 repository. Notebook code cells contain shell/CMake orchestration only—no Python model or data code.


## 1. Clone, build, and test
This checks out the canonical repository and builds the exact C++ implementation that will also load the exported model artifact.


In [ ]:
%%bash
set -euo pipefail
cd /content
rm -rf FruitFly
git clone https://github.com/RyanAteser/FruitFly.git
cd FruitFly
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release
cmake --build build -j2
ctest --test-dir build --output-on-failure
mkdir -p input models/exports results


## 2. Upload frozen inputs
Use the Colab **Files** sidebar to upload these files into `/content/FruitFly/input/`: `phase1.conf`, `btc_events.csv`, `neurons.csv`, `edges.csv`, `sensory.csv`, and `outputs.csv`.

In `phase1.conf`, set `dataset_path=/content/FruitFly/input/btc_events.csv` and `results_dir=/content/FruitFly/results`. Do not change split dates after inspecting validation results unless the experiment is explicitly relabeled exploratory.


In [ ]:
%%bash
set -euo pipefail
cd /content/FruitFly
for f in phase1.conf btc_events.csv neurons.csv edges.csv sensory.csv outputs.csv; do
  test -s "input/$f" || { echo "missing input/$f" >&2; exit 1; }
done
sha256sum input/phase1.conf input/btc_events.csv input/neurons.csv input/edges.csv input/sensory.csv input/outputs.csv


## 3. Train on TRAIN, evaluate VALIDATION, export model
The recurrent fly graph remains fixed. Only the constrained UP/DOWN readout is optimized. This cell intentionally evaluates `validation`; TEST remains locked.


In [ ]:
%%bash
set -euo pipefail
cd /content/FruitFly
./build/flyquant train-connectome \
  --config input/phase1.conf \
  --neurons input/neurons.csv \
  --edges input/edges.csv \
  --sensory input/sensory.csv \
  --outputs input/outputs.csv \
  --model-out models/exports/connectome_reservoir_v1.fqmodel \
  --split validation
sha256sum models/exports/connectome_reservoir_v1.fqmodel


## 4. Reload the exported artifact through the production C++ path
This is an integration check, not a second training run. The loader verifies the neuron and edge source hashes embedded in the model.


In [ ]:
%%bash
set -euo pipefail
cd /content/FruitFly
./build/flyquant predict-connectome \
  --config input/phase1.conf \
  --neurons input/neurons.csv \
  --edges input/edges.csv \
  --model-in models/exports/connectome_reservoir_v1.fqmodel \
  --split validation


## 5. Export to the FlyQuant infrastructure
Download `models/exports/connectome_reservoir_v1.fqmodel` from the Colab Files sidebar. Copy that artifact into the same path (or another controlled model-artifact directory) on the machine running FlyQuant. `predict-connectome` uses the same C++ loader locally and in Colab.

Do **not** run TEST merely because validation looks good. Freeze the protocol and model first; then a deliberate TEST run can use the existing `--allow-test` guard.
